# Combined Sensitivity Analysis

12-panel figure comparing YLL sensitivity (left column), GHG price sensitivity (middle column), and combined GHG+YLL sensitivity (right column).

- Row 1: Food consumption (kcal/person/day)
- Row 2: GHG emissions by food group (GtCO2eq)
- Row 3: Health cost by food group (million YLL)
- Row 4: Objective breakdown (billion USD)

In [ ]:
from pathlib import Path

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import numpy as np
from sensitivity_utils import (
    FONTSIZE_AXIS_LABEL,
    FONTSIZE_PANEL_LABEL,
    FONTSIZE_TICK_LABEL,
    FONTSIZE_TITLE,
    PRETTY_NAMES_HEALTH,
    aggregate_food_groups,
    assign_food_colors,
    extract_combined_scenarios,
    extract_scenarios_with_param,
    get_log_ticks,
    load_consumption_from_statistics,
    load_food_to_group,
    load_ghg_from_statistics,
    load_health_attribution_from_analysis,
    load_net_emissions,
    load_objective_from_analysis,
    log_scale_zero_position,
    plot_objective_sensitivity,
    plot_stacked_emissions,
    plot_stacked_sensitivity,
    prepare_objective_data,
    set_dual_xaxis_labels,
    set_dual_xlabel,
)

In [ ]:
# Configuration
PROJECT_ROOT = Path("..").resolve()

# Single config name for all sensitivity analyses
CONFIG_NAME = "sensitivity"

# Load food to group mapping
FOOD_TO_GROUP = load_food_to_group(PROJECT_ROOT)

# Constants
MIN_FOOD_GROUP_PEAK_SHARE = 0.01  # Drop tiny food groups (<1% of max)

# Production stability cost variants: l1_cost -> scenario name suffix
# The default (0.5) has no suffix; alternatives append e.g. "_l1_0p1"
L1_COST_VARIANTS = {
    0.5: "",
    0.1: "_l1_0p1",
    10: "_l1_10",
}


def filter_by_l1_suffix(scenarios, suffix):
    """Filter scenario tuples to those matching a given l1_cost suffix.

    For the default suffix (""), keeps scenarios WITHOUT any _l1_ in the name.
    For other suffixes (e.g. "_l1_0p1"), keeps scenarios ending with that suffix.
    """
    if suffix == "":
        return [(p, s, f) for p, s, f in scenarios if "_l1_" not in s]
    return [(p, s, f) for p, s, f in scenarios if s.endswith(suffix)]


def filter_combined_by_l1_suffix(scenarios, suffix):
    """Like filter_by_l1_suffix but for 4-tuples (ghg, yll, name, path)."""
    if suffix == "":
        return [(g, y, s, f) for g, y, s, f in scenarios if "_l1_" not in s]
    return [(g, y, s, f) for g, y, s, f in scenarios if s.endswith(suffix)]

## Load YLL Data

In [ ]:
# Extract YLL scenarios from config (default l1_cost only)
yll_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["health", "value_per_yll"],
    scenario_prefix="yll_",
)
yll_scenarios_all = filter_by_l1_suffix(yll_scenarios_all, "")

# Save full config range for x-axis ticks (before filtering to existing files)
yll_param_values_full = [p for p, _, _ in yll_scenarios_all]

# Filter to only include scenarios with existing network files
yll_scenarios = [(p, s, f) for p, s, f in yll_scenarios_all if f.exists()]

print(f"Found {len(yll_scenarios)}/{len(yll_scenarios_all)} YLL scenarios")
yll_param_values = [p for p, _, _ in yll_scenarios]
print(f"YLL values (solved): {yll_param_values}")
if len(yll_scenarios) < len(yll_scenarios_all):
    missing = set(yll_param_values_full) - set(yll_param_values)
    print(f"YLL values (missing): {sorted(missing)}")

In [ ]:
# Extract YLL consumption data from statistics outputs
df_yll_consumption = load_consumption_from_statistics(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="yll_value",
)

# Aggregate and prepare for plotting
df_yll_consumption_plot = aggregate_food_groups(
    df_yll_consumption, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
min_yll = df_yll_consumption_plot.index.min()
yll_group_order = (
    df_yll_consumption_plot.loc[min_yll].sort_values(ascending=False).index.tolist()
)
df_yll_consumption_plot = df_yll_consumption_plot[yll_group_order]
yll_colors = assign_food_colors(df_yll_consumption_plot)

print(f"YLL consumption data shape: {df_yll_consumption_plot.shape}")

In [ ]:
# Extract YLL objective data from precomputed analysis outputs
df_yll_obj = load_objective_from_analysis(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="yll_value",
)
df_yll_obj = prepare_objective_data(df_yll_obj)
print(f"YLL objective data shape: {df_yll_obj.shape}")
print(f"Categories: {df_yll_obj.columns.tolist()}")

In [ ]:
# Extract YLL GHG emissions data from analysis outputs
df_yll_ghg = load_ghg_from_statistics(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="yll_value",
)

# Aggregate and use same order as consumption plot
df_yll_ghg_plot = aggregate_food_groups(
    df_yll_ghg, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
available_groups = [g for g in yll_group_order if g in df_yll_ghg_plot.columns]
df_yll_ghg_plot = df_yll_ghg_plot[available_groups]

print(f"YLL GHG data shape: {df_yll_ghg_plot.shape}")

In [ ]:
# Extract YLL health cost data
df_yll_health = load_health_attribution_from_analysis(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="yll_value",
)

# Aggregate fruits+vegetables to match other panels
df_yll_health_plot = aggregate_food_groups(
    df_yll_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)

# Use available groups that match the health risk factors
yll_health_groups = [g for g in yll_group_order if g in df_yll_health_plot.columns]
df_yll_health_plot = df_yll_health_plot[yll_health_groups]

print(f"YLL health data shape: {df_yll_health_plot.shape}")

## Load GHG Data

In [ ]:
# Extract GHG scenarios from config (default l1_cost only)
ghg_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["emissions", "ghg_price"],
    scenario_prefix="ghg_",
)
# Exclude ghg_yll_ scenarios and non-default l1_cost variants
ghg_scenarios_all = [
    (p, s, f) for p, s, f in ghg_scenarios_all if not s.startswith("ghg_yll_")
]
ghg_scenarios_all = filter_by_l1_suffix(ghg_scenarios_all, "")

# Save full config range for x-axis ticks
ghg_param_values_full = [p for p, _, _ in ghg_scenarios_all]

# Filter to only include scenarios with existing network files
ghg_scenarios = [(p, s, f) for p, s, f in ghg_scenarios_all if f.exists()]

print(f"Found {len(ghg_scenarios)}/{len(ghg_scenarios_all)} GHG scenarios")
ghg_param_values = [p for p, _, _ in ghg_scenarios]
print(f"GHG prices (solved): {ghg_param_values}")
if len(ghg_scenarios) < len(ghg_scenarios_all):
    missing = set(ghg_param_values_full) - set(ghg_param_values)
    print(f"GHG prices (missing): {sorted(missing)}")

In [ ]:
# Extract GHG consumption data from statistics outputs
df_ghg_consumption = load_consumption_from_statistics(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate and use same order as YLL plots (for consistent colors across columns)
df_ghg_consumption_plot = aggregate_food_groups(
    df_ghg_consumption, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
available_groups_ghg = [
    g for g in yll_group_order if g in df_ghg_consumption_plot.columns
]
df_ghg_consumption_plot = df_ghg_consumption_plot[available_groups_ghg]

print(f"GHG consumption data shape: {df_ghg_consumption_plot.shape}")

In [ ]:
# Extract GHG objective data from precomputed analysis outputs
df_ghg_obj = load_objective_from_analysis(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)
df_ghg_obj = prepare_objective_data(df_ghg_obj)
print(f"GHG objective data shape: {df_ghg_obj.shape}")
print(f"Categories: {df_ghg_obj.columns.tolist()}")

In [ ]:
# Extract GHG GHG emissions data from analysis outputs
df_ghg_ghg = load_ghg_from_statistics(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate and use same order as YLL plots (for consistent colors across columns)
df_ghg_ghg_plot = aggregate_food_groups(
    df_ghg_ghg, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
available_groups_ghg_ghg = [g for g in yll_group_order if g in df_ghg_ghg_plot.columns]
df_ghg_ghg_plot = df_ghg_ghg_plot[available_groups_ghg_ghg]

print(f"GHG GHG data shape: {df_ghg_ghg_plot.shape}")

In [ ]:
# Extract GHG health cost data
df_ghg_health = load_health_attribution_from_analysis(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate fruits+vegetables to match other panels
df_ghg_health_plot = aggregate_food_groups(
    df_ghg_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)

# Use available groups that match YLL health data (for consistent colors)
ghg_health_groups = [g for g in yll_health_groups if g in df_ghg_health_plot.columns]
df_ghg_health_plot = df_ghg_health_plot[ghg_health_groups]

print(f"GHG health data shape: {df_ghg_health_plot.shape}")

## Load GHG+YLL Data (Combined)

In [ ]:
# Extract combined GHG+YLL scenarios from config (default l1_cost only)
ghg_yll_scenarios_all = extract_combined_scenarios(
    PROJECT_ROOT,
    CONFIG_NAME,
    ghg_param_path=["emissions", "ghg_price"],
    yll_param_path=["health", "value_per_yll"],
    scenario_prefix="ghg_yll_",
)
ghg_yll_scenarios_all = filter_combined_by_l1_suffix(ghg_yll_scenarios_all, "")

# Save full config range for x-axis ticks
ghg_yll_ghg_values_full = [ghg for ghg, _, _, _ in ghg_yll_scenarios_all]
ghg_yll_yll_values_full = [yll for _, yll, _, _ in ghg_yll_scenarios_all]

# Filter to only include scenarios with existing network files
# Also exclude experimental scenarios (e.g., "break15" variants)
ghg_yll_scenarios_full = [
    (ghg, yll, s, f)
    for ghg, yll, s, f in ghg_yll_scenarios_all
    if f.exists() and "break" not in s
]

# Convert to format expected by extraction functions: (ghg_price, scenario_name, path)
# Use ghg_price as the primary index for plotting
ghg_yll_scenarios = [(ghg, s, f) for ghg, yll, s, f in ghg_yll_scenarios_full]

# Keep track of yll values for dual x-axis labels
ghg_yll_param_pairs = [(ghg, yll) for ghg, yll, s, f in ghg_yll_scenarios_full]

print(f"Found {len(ghg_yll_scenarios)}/{len(ghg_yll_scenarios_all)} GHG_YLL scenarios")
ghg_yll_ghg_values = [ghg for ghg, _ in ghg_yll_param_pairs]
ghg_yll_yll_values = [yll for _, yll in ghg_yll_param_pairs]
print(f"GHG prices (solved): {ghg_yll_ghg_values}")
print(f"YLL values (solved): {ghg_yll_yll_values}")
if len(ghg_yll_scenarios) < len(ghg_yll_scenarios_all):
    missing_ghg = set(ghg_yll_ghg_values_full) - set(ghg_yll_ghg_values)
    print(f"Missing GHG values: {sorted(missing_ghg)}")

In [ ]:
# Extract GHG_YLL consumption data from statistics outputs
df_ghg_yll_consumption = load_consumption_from_statistics(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate and use same order as YLL plots (for consistent colors)
df_ghg_yll_consumption_plot = aggregate_food_groups(
    df_ghg_yll_consumption, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
available_groups_ghg_yll = [
    g for g in yll_group_order if g in df_ghg_yll_consumption_plot.columns
]
df_ghg_yll_consumption_plot = df_ghg_yll_consumption_plot[available_groups_ghg_yll]

print(f"GHG_YLL consumption data shape: {df_ghg_yll_consumption_plot.shape}")

In [ ]:
# Extract GHG_YLL objective data from precomputed analysis outputs
df_ghg_yll_obj = load_objective_from_analysis(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)
df_ghg_yll_obj = prepare_objective_data(df_ghg_yll_obj)
print(f"GHG_YLL objective data shape: {df_ghg_yll_obj.shape}")
print(f"Categories: {df_ghg_yll_obj.columns.tolist()}")

In [ ]:
# Extract GHG_YLL GHG emissions data from analysis outputs
df_ghg_yll_ghg = load_ghg_from_statistics(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate and use same order as YLL plots
df_ghg_yll_ghg_plot = aggregate_food_groups(
    df_ghg_yll_ghg, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)
available_groups_ghg_yll_ghg = [
    g for g in yll_group_order if g in df_ghg_yll_ghg_plot.columns
]
df_ghg_yll_ghg_plot = df_ghg_yll_ghg_plot[available_groups_ghg_yll_ghg]

print(f"GHG_YLL GHG data shape: {df_ghg_yll_ghg_plot.shape}")

In [ ]:
# Extract GHG_YLL health cost data
df_ghg_yll_health = load_health_attribution_from_analysis(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="ghg_price",
)

# Aggregate fruits+vegetables to match other panels
df_ghg_yll_health_plot = aggregate_food_groups(
    df_ghg_yll_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
)

# Use available groups that match YLL health data (for consistent colors)
ghg_yll_health_groups = [
    g for g in yll_health_groups if g in df_ghg_yll_health_plot.columns
]
df_ghg_yll_health_plot = df_ghg_yll_health_plot[ghg_yll_health_groups]

print(f"GHG_YLL health data shape: {df_ghg_yll_health_plot.shape}")

## Combined 12-Panel Figure

In [ ]:
# X-axis configuration - derived from FULL config range (not just solved scenarios)
# This ensures axes cover the intended range even when some scenarios are still solving
YLL_XTICKS, YLL_XTICKLABELS = get_log_ticks(yll_param_values_full)
YLL_XLABEL = "Value per Year of Life Lost [USD/YLL]"

GHG_XTICKS, GHG_XTICKLABELS = get_log_ticks(ghg_param_values_full)
GHG_XLABEL = "GHG price [USD/tCO2eq]"

# Combined GHG+YLL x-axis (both parameters vary together)
GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS = get_log_ticks(ghg_yll_ghg_values_full)
GHG_YLL_GHG_VALUES = list(GHG_YLL_XTICKS)


def _interpolate_yll_for_ghg(ghg_tick, ghg_values, yll_values):
    """Interpolate YLL value for a given GHG tick using log-linear interpolation."""
    ghg_arr = np.array(ghg_values)
    yll_arr = np.array(yll_values)

    # Exact match
    if ghg_tick in ghg_arr:
        idx = np.where(ghg_arr == ghg_tick)[0][0]
        return yll_arr[idx]

    # Find bracketing points (excluding zeros)
    nonzero_mask = ghg_arr > 0
    ghg_nz = ghg_arr[nonzero_mask]
    yll_nz = yll_arr[nonzero_mask]

    if ghg_nz.size == 0:
        return 0.0

    if ghg_tick < ghg_nz.min():
        return yll_nz[0] * (ghg_tick / ghg_nz[0])
    if ghg_tick > ghg_nz.max():
        return yll_nz[-1] * (ghg_tick / ghg_nz[-1])

    idx_upper = np.searchsorted(ghg_nz, ghg_tick)
    idx_lower = idx_upper - 1

    log_ghg_low, log_ghg_high = np.log(ghg_nz[idx_lower]), np.log(ghg_nz[idx_upper])
    log_yll_low, log_yll_high = np.log(yll_nz[idx_lower]), np.log(yll_nz[idx_upper])

    t = (np.log(ghg_tick) - log_ghg_low) / (log_ghg_high - log_ghg_low)
    return np.exp(log_yll_low + t * (log_yll_high - log_yll_low))


# Map tick positions to corresponding YLL values via log-linear interpolation
GHG_YLL_YLL_VALUES = [
    0.0
    if label == "0"
    else _interpolate_yll_for_ghg(
        tick, ghg_yll_ghg_values_full, ghg_yll_yll_values_full
    )
    for tick, label in zip(GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS)
]

print(f"YLL ticks: {YLL_XTICKS} -> {YLL_XTICKLABELS}")
print(f"GHG ticks: {GHG_XTICKS} -> {GHG_XTICKLABELS}")
print(
    f"GHG_YLL ticks: {GHG_YLL_XTICKS} -> GHG {GHG_YLL_GHG_VALUES}, YLL {GHG_YLL_YLL_VALUES}"
)

# Manual label positions for YLL plots
YLL_CONSUMPTION_LABEL_X = {
    "grain": 7,
    "dairy": 30,
    "starchy_vegetable": 5,
    "legumes": 3000,
    "oil": 5,
    "red_meat": 3,
    "sugar": 10,
    "nuts_seeds": 10000,
    "whole_grains": 1000,
    "fruits_vegetables": 300,
    "eggs_poultry": 30,
}

YLL_GHG_LABEL_X = {
    "red_meat": 3,
    "dairy": 30,
    "grain": 7,
    "oil": 10,
    "starchy_vegetable": 5,
    "legumes": 3000,
    "sugar": 5,
    "nuts_seeds": 30000,
    "whole_grains": 500,
    "fruits_vegetables": 3000,
    "eggs_poultry": 30,
}

YLL_OBJ_LABEL_X = {
    "Crop production": 50000,
    "Health burden": 10,
    "GHG cost": 3,
    "Trade": 100000,
    "Consumer values": 5000,
}

# Manual label positions for GHG plots
GHG_CONSUMPTION_LABEL_X = {
    "eggs_poultry": 8,
}

GHG_GHG_LABEL_SKIP = {"legumes", "fruits_vegetables"}

GHG_HEALTH_LABEL_SKIP = {"legumes"}

# GHG_YLL plots - skip small groups
GHG_YLL_LABEL_SKIP = {"legumes"}

In [ ]:
# Configuration option
INCLUDE_OBJECTIVE_ROW = True  # Set to False to hide the objective breakdown row

# Create multipanel figure with shared axes within rows
n_rows = 4 if INCLUDE_OBJECTIVE_ROW else 3
fig_height = 9.5 if INCLUDE_OBJECTIVE_ROW else 7.5
fig, axes = plt.subplots(n_rows, 3, figsize=(10.5, fig_height))

# Row 1: Food consumption
# Panel a: YLL food consumption (top-left)
plot_stacked_sensitivity(
    df_yll_consumption_plot,
    yll_colors,
    axes[0, 0],
    xlabel=YLL_XLABEL,
    ylabel="Food consumption [kcal/person/day]",
    panel_label="a",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
    label_x_positions=YLL_CONSUMPTION_LABEL_X,
    y_max=2400,
)

# Panel b: GHG food consumption (top-middle)
plot_stacked_sensitivity(
    df_ghg_consumption_plot,
    yll_colors,
    axes[0, 1],
    xlabel=GHG_XLABEL,
    ylabel="Food consumption [kcal/person/day]",
    panel_label="b",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    label_x_positions=GHG_CONSUMPTION_LABEL_X,
    y_max=2400,
)

# Panel c: GHG_YLL food consumption (top-right)
plot_stacked_sensitivity(
    df_ghg_yll_consumption_plot,
    yll_colors,
    axes[0, 2],
    xlabel="",  # Will be set with dual labels
    ylabel="Food consumption [kcal/person/day]",
    panel_label="c",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),  # Will be replaced
    label_skip=GHG_YLL_LABEL_SKIP,
    y_max=2400,
)

# Row 2: GHG emissions
# Panel d: YLL GHG emissions
plot_stacked_sensitivity(
    df_yll_ghg_plot,
    yll_colors,
    axes[1, 0],
    xlabel=YLL_XLABEL,
    ylabel="GHG emissions [GtCO2eq]",
    panel_label="d",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
    label_x_positions=YLL_GHG_LABEL_X,
    min_height_for_label=0.08,
)

# Panel e: GHG GHG emissions
plot_stacked_sensitivity(
    df_ghg_ghg_plot,
    yll_colors,
    axes[1, 1],
    xlabel=GHG_XLABEL,
    ylabel="GHG emissions [GtCO2eq]",
    panel_label="e",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    label_skip=GHG_GHG_LABEL_SKIP,
    min_height_for_label=0.08,
)

# Panel f: GHG_YLL GHG emissions
plot_stacked_sensitivity(
    df_ghg_yll_ghg_plot,
    yll_colors,
    axes[1, 2],
    xlabel="",
    ylabel="GHG emissions [GtCO2eq]",
    panel_label="f",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    label_skip=GHG_YLL_LABEL_SKIP,
    min_height_for_label=0.08,
)

# Row 3: Health cost by food group
# Panel g: YLL health cost
plot_stacked_sensitivity(
    df_yll_health_plot,
    yll_colors,
    axes[2, 0],
    xlabel=YLL_XLABEL,
    ylabel="Health cost [million YLL]",
    panel_label="g",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
    min_height_for_label=0.08,
    pretty_names=PRETTY_NAMES_HEALTH,
)

# Panel h: GHG health cost
plot_stacked_sensitivity(
    df_ghg_health_plot,
    yll_colors,
    axes[2, 1],
    xlabel=GHG_XLABEL,
    ylabel="Health cost [million YLL]",
    panel_label="h",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    label_skip=GHG_HEALTH_LABEL_SKIP,
    min_height_for_label=0.08,
    pretty_names=PRETTY_NAMES_HEALTH,
)

# Panel i: GHG_YLL health cost
plot_stacked_sensitivity(
    df_ghg_yll_health_plot,
    yll_colors,
    axes[2, 2],
    xlabel="",
    ylabel="Health cost [million YLL]",
    panel_label="i",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    label_skip=GHG_YLL_LABEL_SKIP,
    min_height_for_label=0.08,
    pretty_names=PRETTY_NAMES_HEALTH,
)

# Row 4: Objective breakdown (optional)
if INCLUDE_OBJECTIVE_ROW:
    # Panel j: YLL objective breakdown
    plot_objective_sensitivity(
        df_yll_obj,
        axes[3, 0],
        xlabel=YLL_XLABEL,
        panel_label="j",
        x_ticks=YLL_XTICKS,
        x_ticklabels=YLL_XTICKLABELS,
        label_x_positions=YLL_OBJ_LABEL_X,
        highlight_cat="GHG cost",
    )

    # Panel k: GHG objective breakdown
    plot_objective_sensitivity(
        df_ghg_obj,
        axes[3, 1],
        xlabel=GHG_XLABEL,
        panel_label="k",
        x_ticks=GHG_XTICKS,
        x_ticklabels=GHG_XTICKLABELS,
        highlight_cat="Health burden",
    )

    # Panel l: GHG_YLL objective breakdown
    plot_objective_sensitivity(
        df_ghg_yll_obj,
        axes[3, 2],
        xlabel="",
        panel_label="l",
        x_ticks=GHG_YLL_XTICKS,
        x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    )

# Add column titles
axes[0, 0].set_title("Health value sensitivity", fontsize=9, fontweight="bold", pad=10)
axes[0, 1].set_title("GHG price sensitivity", fontsize=9, fontweight="bold", pad=10)
axes[0, 2].set_title("Combined sensitivity", fontsize=9, fontweight="bold", pad=10)

# Share y-axis limits within each row
for row in range(n_rows):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-axis labels except for bottom row
for row in range(n_rows - 1):
    for col in range(3):
        axes[row, col].set_xlabel("")
        axes[row, col].set_xticklabels([])

# Set dual-colored x-axis labels for right column (bottom row only)
set_dual_xaxis_labels(
    axes[n_rows - 1, 2],
    GHG_YLL_XTICKS,
    GHG_YLL_GHG_VALUES,
    GHG_YLL_YLL_VALUES,
)
set_dual_xlabel(axes[n_rows - 1, 2])

# Remove y-axis labels from middle and right columns
for row in range(n_rows):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

# Align y-axis labels horizontally across rows
fig.align_ylabels(axes[:, 0])

plt.tight_layout()

# Add extra bottom margin for dual x-axis label on right column
plt.subplots_adjust(bottom=0.08)

# Save to notebooks/figures/
output_dir = PROJECT_ROOT / "notebooks" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "combined_sensitivity.pdf"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path}")

plt.show()

In [ ]:
# Simplified 2-row figure: diet composition + total health/GHG line plots
# Row 1: Diet composition (same as main figure)
# Row 2: Line graphs showing total health cost (MYLL) and total net GHG emissions (GtCO2eq)

from matplotlib.transforms import ScaledTranslation

# Compute total health cost for each sensitivity analysis
yll_total_health = df_yll_health_plot.sum(axis=1)  # MYLL
ghg_total_health = df_ghg_health_plot.sum(axis=1)  # MYLL
ghg_yll_total_health = df_ghg_yll_health_plot.sum(axis=1)  # MYLL

# Load net GHG emissions (includes negative emissions from spared land)
yll_total_ghg = load_net_emissions(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, param_name="yll_value"
)
ghg_total_ghg = load_net_emissions(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, param_name="ghg_price"
)
ghg_yll_total_ghg = load_net_emissions(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, param_name="ghg_price"
)

print(
    f"Net GHG range (YLL sensitivity): {yll_total_ghg.min():.2f} to {yll_total_ghg.max():.2f} GtCO2eq"
)
print(
    f"Net GHG range (GHG sensitivity): {ghg_total_ghg.min():.2f} to {ghg_total_ghg.max():.2f} GtCO2eq"
)
print(
    f"Net GHG range (Combined):        {ghg_yll_total_ghg.min():.2f} to {ghg_yll_total_ghg.max():.2f} GtCO2eq"
)

# Colors for the lines
HEALTH_COLOR = "darkblue"
GHG_COLOR = "darkgreen"

# Spine styling
SPINE_LINEWIDTH = 0.5
SPINE_COLOR = "0.7"

# Figure size: 180mm width, maintain ~2:1 aspect ratio with 2:1 row heights
fig_width_mm = 180
fig_width_in = fig_width_mm / 25.4  # Convert mm to inches
fig_height_in = fig_width_in * 0.52  # Aspect ratio for 2:1 row heights

# Create 2x3 figure with 2:1 height ratio between rows
fig, axes = plt.subplots(
    2, 3, figsize=(fig_width_in, fig_height_in), gridspec_kw={"height_ratios": [2, 1]}
)

# Row 1: Food consumption (same as before)
plot_stacked_sensitivity(
    df_yll_consumption_plot,
    yll_colors,
    axes[0, 0],
    xlabel=YLL_XLABEL,
    ylabel="Food consumption [kcal/person/day]",
    panel_label="a",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
    min_height_for_label=99999,
    y_max=2400,
)

plot_stacked_sensitivity(
    df_ghg_consumption_plot,
    yll_colors,
    axes[0, 1],
    xlabel=GHG_XLABEL,
    ylabel="Food consumption [kcal/person/day]",
    panel_label="b",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=99999,
    y_max=2400,
)

plot_stacked_sensitivity(
    df_ghg_yll_consumption_plot,
    yll_colors,
    axes[0, 2],
    xlabel="",
    ylabel="Food consumption [kcal/person/day]",
    panel_label="c",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    labels_right=True,
    min_height_for_label=0,
    y_max=2400,
)


# Row 2: Line plots with total health cost and net GHG emissions
def plot_totals_line(
    ax,
    health_data,
    ghg_data,
    xlabel,
    panel_label,
    x_ticks,
    x_ticklabels,
    show_legend=False,
):
    """Plot total health cost and net GHG emissions as line graphs with twin y-axes."""
    # Align health and GHG series to common x points (some scenarios may be missing)
    health_aligned = health_data.astype(float)
    ghg_aligned = ghg_data.reindex(health_aligned.index).astype(float)
    valid_mask = health_aligned.notna() & ghg_aligned.notna()
    health_aligned = health_aligned[valid_mask]
    ghg_aligned = ghg_aligned[valid_mask]

    if health_aligned.empty:
        raise ValueError("No overlapping x-values for health and GHG totals")

    x_arr = np.array(health_aligned.index.values, dtype=float)
    zero_pos = log_scale_zero_position(x_arr)
    x_plot = np.where(x_arr == 0, zero_pos, x_arr)

    # Compute xlim matching plot_stacked_sensitivity for alignment
    tick_max = max(x_ticks) if x_ticks else 0
    x_min = zero_pos
    x_max = max(x_plot.max(), tick_max)

    # Plot health cost on primary y-axis
    (line1,) = ax.plot(
        x_plot,
        health_aligned.values,
        color=HEALTH_COLOR,
        linewidth=1,
        marker="o",
        markersize=2,
        label="Health cost",
    )
    ax.set_xscale("log")
    ax.set_ylabel(
        "Health cost\n[million YLL]", fontsize=FONTSIZE_AXIS_LABEL, color=HEALTH_COLOR
    )
    ax.tick_params(axis="y", labelcolor=HEALTH_COLOR, labelsize=FONTSIZE_TICK_LABEL)
    ax.tick_params(axis="x", labelsize=FONTSIZE_TICK_LABEL)

    # Create twin axis for GHG emissions
    ax2 = ax.twinx()
    (line2,) = ax2.plot(
        x_plot,
        ghg_aligned.values,
        color=GHG_COLOR,
        linewidth=1,
        marker="s",
        markersize=2,
        label="Net GHG emissions",
    )
    ax2.set_ylabel(
        "Net GHG emissions\n[GtCO2eq]", fontsize=FONTSIZE_AXIS_LABEL, color=GHG_COLOR
    )
    ax2.tick_params(axis="y", labelcolor=GHG_COLOR, labelsize=FONTSIZE_TICK_LABEL)

    # Add zero reference line for GHG axis
    ax2.axhline(0, color=GHG_COLOR, linewidth=0.5, linestyle="--", alpha=0.5)

    # Set x-axis with same limits as plot_stacked_sensitivity
    ax.set_xlabel(xlabel, fontsize=FONTSIZE_AXIS_LABEL)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_ticklabels)
    ax.set_xlim(x_min, x_max)

    # Panel label (repositioned after tight_layout via absolute offset below)
    ax.text(
        -0.10,
        1.05,
        panel_label,
        transform=ax.transAxes,
        fontsize=FONTSIZE_PANEL_LABEL,
        fontweight="bold",
        va="top",
        ha="left",
    )

    # Grid on primary axis only
    ax.grid(True, alpha=0.3, which="both")
    ax.set_axisbelow(True)

    # Legend - position at top right
    if show_legend:
        lines = [line1, line2]
        labels = [line.get_label() for line in lines]
        ax.legend(
            lines,
            labels,
            loc="upper right",
            fontsize=FONTSIZE_TICK_LABEL,
            framealpha=0.9,
        )

    return ax2


# Panel d: YLL sensitivity - totals
ax2_yll = plot_totals_line(
    axes[1, 0],
    yll_total_health,
    yll_total_ghg,
    xlabel=YLL_XLABEL,
    panel_label="d",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
)

# Panel e: GHG sensitivity - totals
ax2_ghg = plot_totals_line(
    axes[1, 1],
    ghg_total_health,
    ghg_total_ghg,
    xlabel=GHG_XLABEL,
    panel_label="e",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
)

# Panel f: Combined sensitivity - totals (with legend)
ax2_combined = plot_totals_line(
    axes[1, 2],
    ghg_yll_total_health,
    ghg_yll_total_ghg,
    xlabel="",
    panel_label="f",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    show_legend=True,
)

# Add column titles
axes[0, 0].set_title("Health costs priced in", fontsize=FONTSIZE_TITLE, pad=8)
axes[0, 1].set_title("GHG costs priced in", fontsize=FONTSIZE_TITLE, pad=8)
axes[0, 2].set_title("Both priced in simultaneously", fontsize=FONTSIZE_TITLE, pad=8)

# Share y-axis limits for row 1 (consumption)
y_min = min(ax.get_ylim()[0] for ax in axes[0, :])
y_max = max(ax.get_ylim()[1] for ax in axes[0, :])
for col in range(3):
    axes[0, col].set_ylim(y_min, y_max)

# Share y-axis limits for row 2 (health - left axes), starting from 0
health_max = max(axes[1, col].get_ylim()[1] for col in range(3))
for col in range(3):
    axes[1, col].set_ylim(0, health_max)

# Share y-axis limits for row 2 (GHG - right axes)
ghg_min = min(ax.get_ylim()[0] for ax in [ax2_yll, ax2_ghg, ax2_combined])
ghg_max = max(ax.get_ylim()[1] for ax in [ax2_yll, ax2_ghg, ax2_combined])
for ax in [ax2_yll, ax2_ghg, ax2_combined]:
    ax.set_ylim(ghg_min, ghg_max)

# Verify x-axis alignment: both rows must have identical xlim per column
for col in range(3):
    xlim_top = axes[0, col].get_xlim()
    xlim_bot = axes[1, col].get_xlim()
    if xlim_top != xlim_bot:
        print(f"WARNING: col {col} xlim mismatch! top={xlim_top}, bot={xlim_bot}")
        # Force alignment from top row
        axes[1, col].set_xlim(xlim_top)

# Remove x-axis labels from top row
for col in range(3):
    axes[0, col].set_xlabel("")
    axes[0, col].set_xticklabels([])

# Set dual-colored x-axis labels for panel f with custom spacing
# Clear default labels
axes[1, 2].set_xticks(GHG_YLL_XTICKS)
axes[1, 2].set_xticklabels([])
axes[1, 2].set_xlabel("")

# Zero position on the log scale (first tick when label is "0")
ghg_yll_zero_pos = GHG_YLL_XTICKS[0]

# Add dual tick labels manually with increased spacing
trans = axes[1, 2].get_xaxis_transform()
for x, ghg, yll in zip(GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES):
    # Format values - use 0 for the zero position
    if x == ghg_yll_zero_pos:
        ghg_str = "0"
    elif ghg < 1000:
        ghg_str = f"{int(ghg)}"
    else:
        ghg_str = f"{int(ghg/1000)}k"

    if x == ghg_yll_zero_pos:
        yll_str = "0"
    elif yll < 1000:
        yll_str = f"{int(round(yll))}"
    else:
        yll_str = f"{int(round(yll/1000))}k"

    # GHG label (top row of tick labels) - shifted down
    axes[1, 2].text(
        x,
        -0.06,
        ghg_str,
        transform=trans,
        ha="center",
        va="top",
        fontsize=FONTSIZE_TICK_LABEL,
        color=GHG_COLOR,
        fontweight="bold",
    )
    # YLL label (bottom row of tick labels) - shifted down
    axes[1, 2].text(
        x,
        -0.18,
        yll_str,
        transform=trans,
        ha="center",
        va="top",
        fontsize=FONTSIZE_TICK_LABEL,
        color=HEALTH_COLOR,
        fontweight="bold",
    )

# Add dual axis labels with more spacing from tick labels
axes[1, 2].text(
    0.5,
    -0.36,
    "GHG price [USD/tCO2eq]",
    transform=axes[1, 2].transAxes,
    ha="center",
    va="top",
    fontsize=FONTSIZE_AXIS_LABEL,
    color=GHG_COLOR,
)
axes[1, 2].text(
    0.5,
    -0.50,
    "Health value [USD/YLL]",
    transform=axes[1, 2].transAxes,
    ha="center",
    va="top",
    fontsize=FONTSIZE_AXIS_LABEL,
    color=HEALTH_COLOR,
)

# Remove y-axis labels from middle column (for both rows)
axes[0, 1].set_ylabel("")
axes[0, 1].set_yticklabels([])
axes[1, 1].set_ylabel("")
axes[1, 1].set_yticklabels([])

# Remove y-axis labels from right column (for both rows)
axes[0, 2].set_ylabel("")
axes[0, 2].set_yticklabels([])
axes[1, 2].set_ylabel("")
axes[1, 2].set_yticklabels([])

# Keep only right-side y-axis label for rightmost column in row 2
ax2_yll.set_ylabel("")
ax2_yll.set_yticklabels([])
ax2_ghg.set_ylabel("")
ax2_ghg.set_yticklabels([])
# ax2_combined keeps its label

# Style all spines with 0.5 linewidth and grey color
all_axes = [*list(axes.flat), ax2_yll, ax2_ghg, ax2_combined]
for ax in all_axes:
    for spine in ax.spines.values():
        spine.set_linewidth(SPINE_LINEWIDTH)
        spine.set_color(SPINE_COLOR)

plt.tight_layout()
plt.subplots_adjust(bottom=0.22, wspace=0.30, hspace=0.35)

# Reposition all panel labels at a fixed absolute offset from the top-left corner of
# each axes. This gives even spacing regardless of row height: with transAxes coords,
# a 2:1 height ratio would place the upper-row labels twice as far above the axes top.
PANEL_LABEL_DX_PT = -4  # points to the left of the left edge
PANEL_LABEL_DY_PT = 4  # points above the top edge
for ax in axes.flat:
    for txt in list(ax.texts):
        if (
            txt.get_fontsize() == FONTSIZE_PANEL_LABEL
            and txt.get_fontweight() == "bold"
        ):
            label_text = txt.get_text()
            txt.remove()
            offset = ScaledTranslation(
                PANEL_LABEL_DX_PT / 72, PANEL_LABEL_DY_PT / 72, fig.dpi_scale_trans
            )
            ax.text(
                0,
                1,
                label_text,
                transform=ax.transAxes + offset,
                fontsize=FONTSIZE_PANEL_LABEL,
                fontweight="bold",
                va="bottom",
                ha="right",
            )

# Save
output_path_simple = output_dir / "combined_sensitivity_simple.pdf"
plt.savefig(output_path_simple, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path_simple}")
print(f"Figure size: {fig_width_mm}mm x {fig_height_in * 25.4:.1f}mm")

## Multi-page Simple Figure (varying production stability cost)

Generates the simplified 2-row figure for each `l1_cost` variant, saved as a single multi-page PDF.

In [ ]:
from matplotlib.transforms import ScaledTranslation


def load_sensitivity_data(l1_suffix):
    """Load all sensitivity data for a given l1_cost suffix.

    Returns a dict with all data needed for plotting.
    """
    # --- YLL scenarios ---
    yll_all = extract_scenarios_with_param(
        PROJECT_ROOT,
        CONFIG_NAME,
        param_path=["health", "value_per_yll"],
        scenario_prefix="yll_",
    )
    yll_all = filter_by_l1_suffix(yll_all, l1_suffix)
    yll_param_full = [p for p, _, _ in yll_all]
    yll_solved = [(p, s, f) for p, s, f in yll_all if f.exists()]

    df_yll_cons = load_consumption_from_statistics(
        yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="yll_value",
    )
    df_yll_cons_plot = aggregate_food_groups(
        df_yll_cons, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    min_yll = df_yll_cons_plot.index.min()
    group_order = (
        df_yll_cons_plot.loc[min_yll].sort_values(ascending=False).index.tolist()
    )
    df_yll_cons_plot = df_yll_cons_plot[group_order]
    colors = assign_food_colors(df_yll_cons_plot)

    df_yll_health = load_health_attribution_from_analysis(
        yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="yll_value",
    )
    df_yll_health_plot = aggregate_food_groups(
        df_yll_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    yll_health_groups = [g for g in group_order if g in df_yll_health_plot.columns]
    df_yll_health_plot = df_yll_health_plot[yll_health_groups]

    yll_total_health = df_yll_health_plot.sum(axis=1)
    yll_total_ghg = load_net_emissions(
        yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="yll_value",
    )

    # --- GHG scenarios ---
    ghg_all = extract_scenarios_with_param(
        PROJECT_ROOT,
        CONFIG_NAME,
        param_path=["emissions", "ghg_price"],
        scenario_prefix="ghg_",
    )
    ghg_all = [(p, s, f) for p, s, f in ghg_all if not s.startswith("ghg_yll_")]
    ghg_all = filter_by_l1_suffix(ghg_all, l1_suffix)
    ghg_param_full = [p for p, _, _ in ghg_all]
    ghg_solved = [(p, s, f) for p, s, f in ghg_all if f.exists()]

    df_ghg_cons = load_consumption_from_statistics(
        ghg_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )
    df_ghg_cons_plot = aggregate_food_groups(
        df_ghg_cons, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    avail = [g for g in group_order if g in df_ghg_cons_plot.columns]
    df_ghg_cons_plot = df_ghg_cons_plot[avail]

    df_ghg_health = load_health_attribution_from_analysis(
        ghg_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )
    df_ghg_health_plot = aggregate_food_groups(
        df_ghg_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    ghg_health_groups = [
        g for g in yll_health_groups if g in df_ghg_health_plot.columns
    ]
    df_ghg_health_plot = df_ghg_health_plot[ghg_health_groups]

    ghg_total_health = df_ghg_health_plot.sum(axis=1)
    ghg_total_ghg = load_net_emissions(
        ghg_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )

    # --- Combined GHG+YLL scenarios ---
    ghg_yll_all = extract_combined_scenarios(
        PROJECT_ROOT,
        CONFIG_NAME,
        ghg_param_path=["emissions", "ghg_price"],
        yll_param_path=["health", "value_per_yll"],
        scenario_prefix="ghg_yll_",
    )
    ghg_yll_all = filter_combined_by_l1_suffix(ghg_yll_all, l1_suffix)
    ghg_yll_ghg_full = [g for g, _, _, _ in ghg_yll_all]
    ghg_yll_yll_full = [y for _, y, _, _ in ghg_yll_all]

    ghg_yll_full = [
        (g, y, s, f) for g, y, s, f in ghg_yll_all if f.exists() and "break" not in s
    ]
    ghg_yll_solved = [(g, s, f) for g, _, s, f in ghg_yll_full]
    ghg_yll_pairs = [(g, y) for g, y, _, _ in ghg_yll_full]

    df_gy_cons = load_consumption_from_statistics(
        ghg_yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )
    df_gy_cons_plot = aggregate_food_groups(
        df_gy_cons, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    avail_gy = [g for g in group_order if g in df_gy_cons_plot.columns]
    df_gy_cons_plot = df_gy_cons_plot[avail_gy]

    df_gy_health = load_health_attribution_from_analysis(
        ghg_yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )
    df_gy_health_plot = aggregate_food_groups(
        df_gy_health, min_peak_share=MIN_FOOD_GROUP_PEAK_SHARE
    )
    gy_health_groups = [g for g in yll_health_groups if g in df_gy_health_plot.columns]
    df_gy_health_plot = df_gy_health_plot[gy_health_groups]

    gy_total_health = df_gy_health_plot.sum(axis=1)
    gy_total_ghg = load_net_emissions(
        ghg_yll_solved,
        PROJECT_ROOT,
        CONFIG_NAME,
        param_name="ghg_price",
    )

    return {
        # YLL
        "yll_cons": df_yll_cons_plot,
        "yll_total_health": yll_total_health,
        "yll_total_ghg": yll_total_ghg,
        "yll_param_full": yll_param_full,
        "yll_scenarios": yll_solved,
        # GHG
        "ghg_cons": df_ghg_cons_plot,
        "ghg_total_health": ghg_total_health,
        "ghg_total_ghg": ghg_total_ghg,
        "ghg_param_full": ghg_param_full,
        "ghg_scenarios": ghg_solved,
        # Combined
        "gy_cons": df_gy_cons_plot,
        "gy_total_health": gy_total_health,
        "gy_total_ghg": gy_total_ghg,
        "gy_ghg_full": ghg_yll_ghg_full,
        "gy_yll_full": ghg_yll_yll_full,
        "gy_pairs": ghg_yll_pairs,
        "gy_scenarios": ghg_yll_solved,
        # Shared
        "colors": colors,
        "group_order": group_order,
    }


def plot_simple_page(data, suptitle=None):
    """Plot the simplified 2-row figure and return the Figure.

    Replicates the logic from the main simple-figure cell.
    """
    # X-axis ticks
    yll_xticks, yll_xlabels = get_log_ticks(data["yll_param_full"])
    ghg_xticks, ghg_xlabels = get_log_ticks(data["ghg_param_full"])
    gy_xticks, gy_xlabels = get_log_ticks(data["gy_ghg_full"])
    gy_ghg_vals = list(gy_xticks)

    gy_yll_vals = [
        0.0
        if label == "0"
        else _interpolate_yll_for_ghg(tick, data["gy_ghg_full"], data["gy_yll_full"])
        for tick, label in zip(gy_xticks, gy_xlabels)
    ]

    health_color = "darkblue"
    ghg_color = "darkgreen"
    spine_lw = 0.5
    spine_c = "0.7"

    fig_w = 180 / 25.4
    fig_h = fig_w * 0.52
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(fig_w, fig_h),
        gridspec_kw={"height_ratios": [2, 1]},
    )

    colors = data["colors"]

    # Row 1: consumption
    plot_stacked_sensitivity(
        data["yll_cons"],
        colors,
        axes[0, 0],
        xlabel="Value per Year of Life Lost [USD/YLL]",
        ylabel="Food consumption [kcal/person/day]",
        panel_label="a",
        x_ticks=yll_xticks,
        x_ticklabels=yll_xlabels,
        min_height_for_label=99999,
        y_max=2400,
    )
    plot_stacked_sensitivity(
        data["ghg_cons"],
        colors,
        axes[0, 1],
        xlabel="GHG price [USD/tCO2eq]",
        ylabel="Food consumption [kcal/person/day]",
        panel_label="b",
        x_ticks=ghg_xticks,
        x_ticklabels=ghg_xlabels,
        min_height_for_label=99999,
        y_max=2400,
    )
    plot_stacked_sensitivity(
        data["gy_cons"],
        colors,
        axes[0, 2],
        xlabel="",
        ylabel="Food consumption [kcal/person/day]",
        panel_label="c",
        x_ticks=gy_xticks,
        x_ticklabels=[""] * len(gy_xticks),
        labels_right=True,
        min_height_for_label=0,
        y_max=2400,
    )

    # Row 2: total health + GHG lines
    def _plot_totals(ax, health, ghg, xlabel, label, xticks, xlabels, legend=False):
        h = health.astype(float)
        g = ghg.reindex(h.index).astype(float)
        mask = h.notna() & g.notna()
        h, g = h[mask], g[mask]
        x = np.array(h.index.values, dtype=float)
        zp = log_scale_zero_position(x)
        xp = np.where(x == 0, zp, x)
        tick_max = max(xticks) if xticks else 0

        (l1,) = ax.plot(
            xp,
            h.values,
            color=health_color,
            lw=1,
            marker="o",
            ms=2,
            label="Health cost",
        )
        ax.set_xscale("log")
        ax.set_ylabel(
            "Health cost\n[million YLL]",
            fontsize=FONTSIZE_AXIS_LABEL,
            color=health_color,
        )
        ax.tick_params(axis="y", labelcolor=health_color, labelsize=FONTSIZE_TICK_LABEL)
        ax.tick_params(axis="x", labelsize=FONTSIZE_TICK_LABEL)

        ax2 = ax.twinx()
        (l2,) = ax2.plot(
            xp,
            g.values,
            color=ghg_color,
            lw=1,
            marker="s",
            ms=2,
            label="Net GHG emissions",
        )
        ax2.set_ylabel(
            "Net GHG emissions\n[GtCO2eq]",
            fontsize=FONTSIZE_AXIS_LABEL,
            color=ghg_color,
        )
        ax2.tick_params(axis="y", labelcolor=ghg_color, labelsize=FONTSIZE_TICK_LABEL)
        ax2.axhline(0, color=ghg_color, lw=0.5, ls="--", alpha=0.5)

        ax.set_xlabel(xlabel, fontsize=FONTSIZE_AXIS_LABEL)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xlabels)
        ax.set_xlim(zp, max(xp.max(), tick_max))
        ax.text(
            -0.10,
            1.05,
            label,
            transform=ax.transAxes,
            fontsize=FONTSIZE_PANEL_LABEL,
            fontweight="bold",
            va="top",
            ha="left",
        )
        ax.grid(True, alpha=0.3, which="both")
        ax.set_axisbelow(True)

        if legend:
            ax.legend(
                [l1, l2],
                [l1.get_label(), l2.get_label()],
                loc="upper right",
                fontsize=FONTSIZE_TICK_LABEL,
                framealpha=0.9,
            )
        return ax2

    ax2_y = _plot_totals(
        axes[1, 0],
        data["yll_total_health"],
        data["yll_total_ghg"],
        "Value per Year of Life Lost [USD/YLL]",
        "d",
        yll_xticks,
        yll_xlabels,
    )
    ax2_g = _plot_totals(
        axes[1, 1],
        data["ghg_total_health"],
        data["ghg_total_ghg"],
        "GHG price [USD/tCO2eq]",
        "e",
        ghg_xticks,
        ghg_xlabels,
    )
    ax2_c = _plot_totals(
        axes[1, 2],
        data["gy_total_health"],
        data["gy_total_ghg"],
        "",
        "f",
        gy_xticks,
        [""] * len(gy_xticks),
        legend=True,
    )

    # Titles
    axes[0, 0].set_title("Health costs priced in", fontsize=FONTSIZE_TITLE, pad=8)
    axes[0, 1].set_title("GHG costs priced in", fontsize=FONTSIZE_TITLE, pad=8)
    axes[0, 2].set_title(
        "Both priced in simultaneously", fontsize=FONTSIZE_TITLE, pad=8
    )

    if suptitle:
        fig.suptitle(suptitle, fontsize=FONTSIZE_TITLE + 1, fontweight="bold", y=1.02)

    # Share y-limits row 1
    ym = max(ax.get_ylim()[1] for ax in axes[0, :])
    yn = min(ax.get_ylim()[0] for ax in axes[0, :])
    for c in range(3):
        axes[0, c].set_ylim(yn, ym)

    # Share y-limits row 2 (health left)
    hm = max(axes[1, c].get_ylim()[1] for c in range(3))
    for c in range(3):
        axes[1, c].set_ylim(0, hm)

    # Share y-limits row 2 (GHG right)
    gn = min(a.get_ylim()[0] for a in [ax2_y, ax2_g, ax2_c])
    gx = max(a.get_ylim()[1] for a in [ax2_y, ax2_g, ax2_c])
    for a in [ax2_y, ax2_g, ax2_c]:
        a.set_ylim(gn, gx)

    # Align x-limits
    for c in range(3):
        axes[1, c].set_xlim(axes[0, c].get_xlim())

    # Remove x labels from top row
    for c in range(3):
        axes[0, c].set_xlabel("")
        axes[0, c].set_xticklabels([])

    # Dual x-axis for panel f
    axes[1, 2].set_xticks(gy_xticks)
    axes[1, 2].set_xticklabels([])
    axes[1, 2].set_xlabel("")
    zp = gy_xticks[0]
    trans = axes[1, 2].get_xaxis_transform()
    for x, ghg, yll in zip(gy_xticks, gy_ghg_vals, gy_yll_vals):
        gs = "0" if x == zp else (f"{int(ghg)}" if ghg < 1000 else f"{int(ghg/1000)}k")
        ys = (
            "0"
            if x == zp
            else (f"{int(round(yll))}" if yll < 1000 else f"{int(round(yll/1000))}k")
        )
        axes[1, 2].text(
            x,
            -0.06,
            gs,
            transform=trans,
            ha="center",
            va="top",
            fontsize=FONTSIZE_TICK_LABEL,
            color=ghg_color,
            fontweight="bold",
        )
        axes[1, 2].text(
            x,
            -0.18,
            ys,
            transform=trans,
            ha="center",
            va="top",
            fontsize=FONTSIZE_TICK_LABEL,
            color=health_color,
            fontweight="bold",
        )
    axes[1, 2].text(
        0.5,
        -0.36,
        "GHG price [USD/tCO2eq]",
        transform=axes[1, 2].transAxes,
        ha="center",
        va="top",
        fontsize=FONTSIZE_AXIS_LABEL,
        color=ghg_color,
    )
    axes[1, 2].text(
        0.5,
        -0.50,
        "Health value [USD/YLL]",
        transform=axes[1, 2].transAxes,
        ha="center",
        va="top",
        fontsize=FONTSIZE_AXIS_LABEL,
        color=health_color,
    )

    # Remove y labels from middle/right columns
    for r in range(2):
        for c in [1, 2]:
            axes[r, c].set_ylabel("")
            axes[r, c].set_yticklabels([])
    ax2_y.set_ylabel("")
    ax2_y.set_yticklabels([])
    ax2_g.set_ylabel("")
    ax2_g.set_yticklabels([])

    # Style spines
    for ax in [*list(axes.flat), ax2_y, ax2_g, ax2_c]:
        for sp in ax.spines.values():
            sp.set_linewidth(spine_lw)
            sp.set_color(spine_c)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.22, wspace=0.30, hspace=0.35)

    # Reposition panel labels
    dx, dy = -4, 4
    for ax in axes.flat:
        for txt in list(ax.texts):
            if (
                txt.get_fontsize() == FONTSIZE_PANEL_LABEL
                and txt.get_fontweight() == "bold"
            ):
                lab = txt.get_text()
                txt.remove()
                off = ScaledTranslation(dx / 72, dy / 72, fig.dpi_scale_trans)
                ax.text(
                    0,
                    1,
                    lab,
                    transform=ax.transAxes + off,
                    fontsize=FONTSIZE_PANEL_LABEL,
                    fontweight="bold",
                    va="bottom",
                    ha="right",
                )

    return fig


# Generate multi-page PDF
output_dir = PROJECT_ROOT / "notebooks" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path_multi = output_dir / "combined_sensitivity_simple.pdf"

with PdfPages(output_path_multi) as pdf:
    for l1_cost, suffix in L1_COST_VARIANTS.items():
        print(f"\n--- l1_cost = {l1_cost} (suffix: {suffix!r}) ---")
        data = load_sensitivity_data(suffix)
        n_solved = (
            len(data["yll_scenarios"])
            + len(data["ghg_scenarios"])
            + len(data["gy_scenarios"])
        )
        print(f"Total solved scenarios: {n_solved}")
        if n_solved == 0:
            print("No solved scenarios, skipping page.")
            continue
        fig = plot_simple_page(data, suptitle=f"Production stability cost = {l1_cost}")
        pdf.savefig(fig, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

print(f"\nSaved multi-page PDF to: {output_path_multi}")

## Consumption deviation vs baseline


In [ ]:
# New figure: stacked deviations in food-group consumption relative to scen-baseline
baseline_scenarios = [(0.0, "baseline", Path(""))]

# Load baseline and scenario consumption (kcal/person/day), keeping all groups
baseline_consumption = load_consumption_from_statistics(
    baseline_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    param_name="baseline",
)
baseline_consumption = aggregate_food_groups(
    baseline_consumption,
    min_peak_share=0.0,
)

yll_consumption_all = aggregate_food_groups(
    df_yll_consumption,
    min_peak_share=0.0,
)
ghg_consumption_all = aggregate_food_groups(
    df_ghg_consumption,
    min_peak_share=0.0,
)
ghg_yll_consumption_all = aggregate_food_groups(
    df_ghg_yll_consumption,
    min_peak_share=0.0,
)

baseline_series = baseline_consumption.iloc[0]

group_order = baseline_series.sort_values(ascending=False).index.tolist()


def align_groups(df):
    return df.reindex(columns=group_order, fill_value=0.0)


baseline_aligned = baseline_series.reindex(group_order, fill_value=0.0)
yll_consumption_all = align_groups(yll_consumption_all)
ghg_consumption_all = align_groups(ghg_consumption_all)
ghg_yll_consumption_all = align_groups(ghg_yll_consumption_all)

# Deviations relative to baseline scenario
yll_dev = yll_consumption_all.subtract(baseline_aligned, axis=1)
ghg_dev = ghg_consumption_all.subtract(baseline_aligned, axis=1)
ghg_yll_dev = ghg_yll_consumption_all.subtract(baseline_aligned, axis=1)

# Optionally drop tiny groups (by peak absolute deviation across all panels)
max_abs_dev = yll_dev.abs().max(axis=0)
max_abs_dev = max_abs_dev.combine(ghg_dev.abs().max(axis=0), max)
max_abs_dev = max_abs_dev.combine(ghg_yll_dev.abs().max(axis=0), max)
significant_groups = max_abs_dev[max_abs_dev >= 1.0].index.tolist()
if significant_groups:
    group_order = [g for g in group_order if g in significant_groups]

yll_dev = yll_dev.reindex(columns=group_order, fill_value=0.0)
ghg_dev = ghg_dev.reindex(columns=group_order, fill_value=0.0)
ghg_yll_dev = ghg_yll_dev.reindex(columns=group_order, fill_value=0.0)

# Colors (same mapping used elsewhere in the notebook)
baseline_for_colors = baseline_consumption.reindex(columns=group_order)
dev_colors = assign_food_colors(baseline_for_colors)

fig, axes = plt.subplots(1, 3, figsize=(7.1, 3.2), sharey=True)

plot_stacked_emissions(
    yll_dev,
    dev_colors,
    axes[0],
    xlabel=YLL_XLABEL,
    ylabel="Consumption deviation\n[kcal/person/day]",
    panel_label="a",
    x_ticks=YLL_XTICKS,
    x_ticklabels=YLL_XTICKLABELS,
    min_height_for_label=1e9,
)
plot_stacked_emissions(
    ghg_dev,
    dev_colors,
    axes[1],
    xlabel=GHG_XLABEL,
    ylabel="",
    panel_label="b",
    x_ticks=GHG_XTICKS,
    x_ticklabels=GHG_XTICKLABELS,
    min_height_for_label=1e9,
)
plot_stacked_emissions(
    ghg_yll_dev,
    dev_colors,
    axes[2],
    xlabel="",
    ylabel="",
    panel_label="c",
    x_ticks=GHG_YLL_XTICKS,
    x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    min_height_for_label=1e9,
)

axes[0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE, fontweight="bold", pad=8
)
axes[1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE, fontweight="bold", pad=8
)
axes[2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE, fontweight="bold", pad=8
)

set_dual_xaxis_labels(
    axes[2],
    GHG_YLL_XTICKS,
    GHG_YLL_GHG_VALUES,
    GHG_YLL_YLL_VALUES,
)
set_dual_xlabel(axes[2])

# Harmonize y-limits across panels
y_min = min(ax.get_ylim()[0] for ax in axes)
y_max = max(ax.get_ylim()[1] for ax in axes)
for ax in axes:
    ax.set_ylim(y_min, y_max)

# Remove left labels from middle/right panels
for col in [1, 2]:
    axes[col].set_ylabel("")
    axes[col].tick_params(axis="y", labelleft=False)

axes[0].tick_params(axis="y", labelleft=True)

legend_handles = [
    plt.Line2D([0], [0], color=dev_colors[g], lw=2.0, label=g)
    for g in group_order
    if g in dev_colors
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=FONTSIZE_TICK_LABEL,
    bbox_to_anchor=(0.5, -0.04),
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.22)

output_dir = PROJECT_ROOT / "notebooks" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path_consumption = output_dir / "consumption_deviation.pdf"
plt.savefig(output_path_consumption, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path_consumption}")

plt.show()

In [ ]:
plt.subplots_adjust(wspace=0.15, hspace=0.30, bottom=0.16)